# 🔎 Scientific Literature Search — Colab Training (H100 AUTOPILOT, resume-safe)

Fine-tunes the **dense retriever** (bge-small bi-encoder, MultipleNegativesRankingLoss), evaluates
**nDCG@10 / Recall@k / MRR@10** vs the BM25 and zero-shot baselines, runs the **agent**, and generates
`report.pdf` + `slides.pptx`.

**Auto-adapts** to H100 / A100 / L4 / T4 (batch size = number of in-batch negatives; precision) and
**resumes** from the last checkpoint on Drive. Set the Controls and `Runtime → Run all`.


In [ ]:
#@title 0) Controls — set these, then `Runtime → Run all`  { display-mode: "form" }
GIT_REPO_URL = "" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
PROJECT_DIR_NAME = "08_Scientific_Literature_Search" #@param {type:"string"}
USE_DRIVE = True #@param {type:"boolean"}
DRIVE_SUBDIR = "scisearch" #@param {type:"string"}
BASE_MODEL = "auto" #@param ["auto", "BAAI/bge-small-en-v1.5", "sentence-transformers/all-MiniLM-L6-v2", "malteos/scincl"]
CORPUS_LIMIT = 20000 #@param {type:"integer"}
MAX_TRAIN_PAIRS = 80000 #@param {type:"integer"}
EPOCHS = 1 #@param {type:"integer"}
print("Controls set. Repo:", GIT_REPO_URL or "(will copy from Drive)")

In [ ]:
#@title 1) Check the GPU
!nvidia-smi -L || echo "No GPU — Runtime > Change runtime type > GPU"

In [ ]:
#@title 2) Mount Drive + artifact paths & HF caches  (BEFORE importing torch)
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ART = f"/content/drive/MyDrive/{DRIVE_SUBDIR}"
else:
    ART = "/content/scisearch"
os.makedirs(ART, exist_ok=True)
os.environ["SCISEARCH_ARTIFACTS_DIR"] = ART
os.environ["SCISEARCH_MODEL_DIR"] = f"{ART}/models"
os.environ["SCISEARCH_DATA_DIR"]  = f"{ART}/data"
os.environ["SCISEARCH_INDEX_DIR"] = f"{ART}/index"
os.environ["SCISEARCH_RUN_DIR"]   = f"{ART}/runs"
os.environ["HF_HOME"] = f"{ART}/hf_cache"
print("Artifacts (Drive-persisted) ->", ART)

In [ ]:
#@title 3) Get the project source (git clone, or copy from Drive)
import os
WORK = "/content/" + PROJECT_DIR_NAME
if GIT_REPO_URL:
    if not os.path.exists(WORK):
        rc = os.system(f'git clone -b "{BRANCH}" "{GIT_REPO_URL}" "{WORK}"')
        if rc != 0:
            os.system(f'git clone "{GIT_REPO_URL}" "{WORK}"')
else:
    src = f"/content/drive/MyDrive/{PROJECT_DIR_NAME}"
    assert os.path.exists(src), f"Set GIT_REPO_URL or upload the project to Drive/{PROJECT_DIR_NAME}"
    os.system(f'cp -r "{src}" "{WORK}"')
os.chdir(WORK)
print("cwd:", os.getcwd())

In [ ]:
#@title 4) Install dependencies (Colab-safe: NEVER reinstall torch)
%cd $WORK
!pip -q install -r requirements_colab.txt
!pip -q install -e . --no-deps
print("Dependencies installed (torch untouched).")

In [ ]:
#@title 5) Verify environment + performance knobs (TF32)
import torch, sentence_transformers
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("torch", torch.__version__, "| sentence-transformers", sentence_transformers.__version__)
print("GPU:", GPU, "| CUDA:", torch.cuda.is_available())

In [ ]:
#@title 6) Auto GPU profile (batch size = number of in-batch negatives)
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
prof = {"base_model": "BAAI/bge-small-en-v1.5"}
if "H100" in GPU:   bs, bf16, fp16 = 256, True,  False
elif "A100" in GPU: bs, bf16, fp16 = 128, True,  False
elif "L4" in GPU:   bs, bf16, fp16 = 64,  True,  False
elif "T4" in GPU:   bs, bf16, fp16 = 32,  False, True
else:               bs, bf16, fp16 = 16,  False, False
if BASE_MODEL != "auto":
    prof["base_model"] = BASE_MODEL
prof.update(per_device_train_batch_size=bs, bf16=bf16, fp16=fp16)
print(GPU, "->", prof)

In [ ]:
#@title 7) Write the Colab training config  (configs/train_colab.yaml)
import yaml
cfg = {
    "data": {"hf_corpus": "gfissore/arxiv-abstracts-2021", "use_hf_corpus": True,
             "corpus_limit": int(CORPUS_LIMIT), "max_train_pairs": int(MAX_TRAIN_PAIRS),
             "eval_queries": 2000, "seed": 42},
    "model": {**prof, "num_train_epochs": int(EPOCHS), "learning_rate": 2.0e-5,
              "max_seq_length": 256, "warmup_ratio": 0.1, "tf32": True, "eval_steps": 500, "save_steps": 500},
    "reranker": {"model": "cross-encoder/ms-marco-MiniLM-L-6-v2", "enabled": True, "top_k_rerank": 30},
}
open("configs/train_colab.yaml", "w").write(yaml.safe_dump(cfg, sort_keys=False))
print(open("configs/train_colab.yaml").read())

In [ ]:
#@title 8) Build corpus + preview generated (query, paper) pairs
!scisearch --config configs/train_colab.yaml data
!scisearch --config configs/train_colab.yaml pairs

## ⭐ ONE BUTTON — autopilot (resume-safe)
Fine-tunes the retriever → evaluates (vs BM25 + zero-shot) → analysis → `report.pdf` + `slides.pptx`.
**Re-run after a disconnect to RESUME** from the last checkpoint on Drive.

In [ ]:
#@title 9) ⭐ ONE BUTTON autopilot  (re-run to resume)
!scisearch --config configs/train_colab.yaml autopilot

## Individual steps (optional) — idempotent + resume-safe

In [ ]:
#@title 10a) Train only (resumes from the last checkpoint)
!scisearch --config configs/train_colab.yaml train

In [ ]:
#@title 10b) Evaluate — retriever vs BM25 + zero-shot (Recall/MRR/nDCG)
!scisearch --config configs/train_colab.yaml evaluate

In [ ]:
#@title 11) Diagnostics: eval metrics + model metadata
import json, os
rd = os.environ["SCISEARCH_RUN_DIR"]
try:
    ev = json.load(open(f"{rd}/eval/latest.json"))
    print("SUMMARY:", json.dumps(ev.get("summary", {}), indent=2))
    print("model:", ev.get("model"), "\nbm25:", ev.get("bm25"), "\nzero-shot:", ev.get("zero_shot_base"))
except Exception as e:
    print("run evaluate first:", e)

## ✅ Test the trained model

In [ ]:
#@title 12) Search with the trained retriever (results + facets + clusters)
from scisearch.config import load_config
from scisearch.agent.search_agent import SearchAgent
agent = SearchAgent(load_config("configs/train_colab.yaml"), load_model=True)
for q in ["contrastive learning for sentence embeddings",
          "retrieval augmented generation for question answering",
          "efficient attention for long documents"]:
    sd = agent.search(q, save=False).to_dict()
    print(f"\nQUERY: {q!r}  (intent={sd['intent']}, decisions={[(d['id'],d['branch']) for d in sd['decisions']]})")
    print("  facets:", [(f['field'], f['count']) for f in sd['facets'][:4]])
    print("  clusters:", [c['label'] for c in sd['clusters']][:4])
    for r in sd["results"][:5]:
        print(f"    [{r['score']:.3f}] {r['title'][:70]}")

In [ ]:
#@title 13) Locate deliverables (report.pdf + slides.pptx + bundle)
import glob, os
sub = sorted(glob.glob(os.environ["SCISEARCH_ARTIFACTS_DIR"] + "/submission/*"))
if sub:
    print("Submission bundle ->", sub[-1])
    for f in sorted(glob.glob(sub[-1] + "/*")):
        print("  ", os.path.basename(f), os.path.getsize(f), "bytes")
else:
    print("Run the autopilot cell (9) to produce report.pdf + slides.pptx.")

In [ ]:
#@title 14) (Optional) Serve the API + Gradio UI
# !scisearch serve --ui --port 7860
print("Uncomment the line above to launch the FastAPI + Gradio search app.")

## ✅ Final checklist
- [ ] GPU profile picked (cell 6) — H100 bs256 → A100 bs128 → L4 bs64 → T4 bs32 fp16
- [ ] Corpus built (cell 8), training finished (cell 9 / 10a)
- [ ] `evaluate` shows the retriever **beats BM25 + zero-shot** on nDCG@10 (cell 10b / 11)
- [ ] Search returns relevant papers + facets + clusters (cell 12)
- [ ] `report.pdf` + `slides.pptx` produced (cell 13)
- [ ] Everything persisted to Drive (re-run cell 9 to resume after a disconnect)
